# CyberAccountant pipeline — interactive driver

Run cells top to bottom. Each section runs one stage and shows its artifact.

Config: change `ID_STYLE` between `"mnemonic"` and `"numeric"` to compare,
and `OBFUSCATE_TYPES` to hide exact values of chosen entity types from the
relations LLM (forcing it to write equations instead of judging numbers).

In [ ]:
from pathlib import Path
from checkier import Pipeline, PipelineConfig

ID_STYLE = "mnemonic"          # or "numeric"
TAG_MODEL = "deepseek/deepseek-v4.1-flash-20261017"
# NB: the old "~deepseek/deepseek-pro-latest" id 404s on this account —
# relations run fine on the flash model too.
REL_MODEL = TAG_MODEL
# Hide exact values of these entity types from the relations LLM, so it must
# write equations instead of judging the numbers itself. The solver still
# evaluates against the real values. () disables obfuscation.
OBFUSCATE_TYPES = ("amount", "number")

cfg = PipelineConfig(
    data_dir=Path("../data_md"),
    out_dir=Path("../out"),
    id_style=ID_STYLE,
    tag_model=TAG_MODEL,
    rel_model=REL_MODEL,
    obfuscate_types=OBFUSCATE_TYPES,
)
p = Pipeline(cfg)

## Stage 0 — ingest / rename / dedup

In [2]:
mapping = p.ingest()
import pandas as pd
pd.DataFrame(mapping).T

,original,role,sha256,duplicate_of,tag
eb920ab6,данные/02_01_Ведомость_ЗП_Январь_2026.xlsx.md,data,4aa8154585ee1c226d3434b25fed004d2463a6e5e5c195...,NaN,NaN
ddad231f,данные/02_02_Ведомость_ЗП_Февраль_2026.xlsx.md,data,c4ccaff6c874f63c26af97d4de448f9a0dc3ac1f811349...,NaN,NaN
5eda16da,данные/02_03_Ведомость_ЗП_Март_2026.xlsx.md,data,5e25554b83e1c434ef4c6befdc0b43ebba05f8ce83bdd9...,NaN,NaN
7c62dfe6,данные/02_04_Ведомость_ЗП_Апрель_2026.xlsx.md,data,4e108b7cc42f3d987d779e1c255bc3403ed969cde1c52e...,NaN,NaN
819ec2c2,данные/02_05_Ведомость_ЗП_Май_2026.xlsx.md,data,823f89a747de7db43d6a8c25ee98cb00b92bea18879527...,NaN,NaN
59910ba2,данные/03_01_Страховые_взносы_Январь_2026.xlsx.md,data,f577cc50ec3cebb0f47b7a5f14f511871dd4e3723af3a8...,NaN,NaN
080cd35a,данные/03_02_Страховые_взносы_Февраль_2026.xls...,data,77d1da207f88b9eef815aabfb42b2e7c6f5362403d1007...,NaN,NaN
2c5a26f6,данные/03_03_Страховые_взносы_Март_2026.xlsx.md,data,cb481eef1a047a8cecf7d255b900efbe9c73a4bf48181a...,NaN,NaN
b2326f1a,данные/03_04_Страховые_взносы_Апрель_2026.xlsx.md,data,b988a7139882f844ed169c3f7dc0c5c64896f56bd30411...,NaN,NaN
6314a3a9,данные/03_05_Страховые_взносы_Май_2026.xlsx.md,data,8414801ec16918fd770ae8505a267e967480b80c13d94c...,NaN,NaN


## Stage 1 — entity tagging (LLM, per file)

In [3]:
from checkier.stage1_tag import TagProgress, TagStats


def on_tag_progress(prog: TagProgress, stats: TagStats) -> None:
    """Swappable live-progress hook (forward to a frontend here later)."""
    if prog.ok:
        line = (
            f"[{stats.done}/{stats.total}] OK  {prog.file}  "
            f"entities={prog.entities} retries={prog.retries}"
        )
    else:
        line = f"[{stats.done}/{stats.total}] ERR {prog.file}  {prog.error}"
    print(line, flush=True)


tagged = p.tag(on_progress=on_tag_progress)
print(f"\nDone: {len(tagged)} files tagged.")
list(tagged.keys())[:5], len(tagged)

[1/24] OK  2c5a26f6  entities=6 retries=0
[2/24] OK  080cd35a  entities=6 retries=0
[3/24] OK  ddad231f  entities=9 retries=0
[4/24] OK  b2326f1a  entities=6 retries=0
[5/24] OK  75c2d8bf  entities=6 retries=0
[6/24] OK  819ec2c2  entities=8 retries=1
[7/24] OK  6314a3a9  entities=6 retries=0
[8/24] OK  7c62dfe6  entities=8 retries=1
[9/24] OK  eb920ab6  entities=9 retries=1
[10/24] ERR 59910ba2  tagging failed for 59910ba2: rebuilt doc diverges at char 146: expected '\\| Период: Январь 2026 |  |  |  |\n|  |  ', got '| Период: Январь 2026 |  |  |  |\n|  |  |'
[11/24] OK  5eda16da  entities=9 retries=1
[12/24] OK  debeb73e  entities=5 retries=0
[13/24] OK  fca32a8d  entities=4 retries=0
[14/24] OK  f4cb9010  entities=6 retries=0
[15/24] OK  ef8c5d7b  entities=4 retries=0
[16/24] OK  822de338  entities=5 retries=0
[17/24] OK  4a441300  entities=4 retries=0
[18/24] OK  bcb11054  entities=4 retries=0
[19/24] OK  edf020d9  entities=5 retries=0
[20/24] OK  2700be8d  entities=5 retries=0
[21/

(['2c5a26f6', '080cd35a', 'ddad231f', 'b2326f1a', '75c2d8bf'], 23)

## Stage 2 — global entity registry

In [4]:
reg = p.registry()
import pandas as pd
rows = [{"num": e.num, "mnemonic": e.mnemonic, "type": e.type,
         "raw": e.raw, "value": e.value, "unit": e.unit, "file": e.file,
         "role": e.role} for e in reg.entities]
df = pd.DataFrame(rows)
df.head(20)

/home/alice/code/shitposting/CyberAccountant/server/checkier/bip39.py:90: RuntimeWarning: BIP39 wordlist not found; using deterministic generated fallback. For the canonical list run: python scripts/fetch_bip39.py
  self._a, self._b = load_words()


,num,mnemonic,type,raw,value,unit,file,role
0,0,dere-gidy,docid,ВЗН-2026-02,ВЗН-2026-02,,080cd35a,data
1,1,cusa-zypu,contract,24/03/15-01,24/03/15-01,,080cd35a,data
2,2,vybi-jiku,date,Февраль 2026,None,,080cd35a,data
3,3,nyce-jamo,amount,"1,117,000.00",1117000.00,RUB,080cd35a,data
4,4,hela-gaqu,amount,"335,100.00",335100.00,RUB,080cd35a,data
5,5,vexa-refo,number,0300,300,,080cd35a,data
6,6,mala-siry,docid,3/1,3/1,,0b8a21fe,card
7,7,qeba-faqu,date,Август 2025,None,,0b8a21fe,card
8,8,rade-gydo,date,Май 2026,None,,0b8a21fe,card
9,9,fubi-basy,number,16,16,,0b8a21fe,card


In [5]:
df["type"].value_counts()

type
amount      83
number      59
date        34
contract    24
docid       22
Name: count, dtype: int64

## Stage 2b — date normalization (LLM, one batch call)

The deterministic parser handles `DD.MM.YYYY` and `DD month YYYY`, but not
period labels like `Январь 2026` — those entities get `value = None` and
would stay unknowns in the solver. This stage asks the model for concrete
dates (`Январь 2026` → `01.01.2026`, first day of the period). Provenance
is kept in `Entity.normalized_from`.

In [6]:
changed = p.normalize_dates()
print(f"normalized {len(changed)} date entities")
[(e.mnemonic, e.normalized_from, e.value) for e in changed]

normalized 19 date entities


[('vybi-jiku', 'Февраль 2026', 739648),
 ('qeba-faqu', 'Август 2025', 739464),
 ('rade-gydo', 'Май 2026', 739737),
 ('loqa-peny', 'августа 2025', 739464),
 ('dyxi-zalo', 'мая 2026', 739737),
 ('ruwu-jabe', 'Март 2026', 739676),
 ('wogo-kiqi', 'Март_2026', 739676),
 ('hedi-jajo', 'Март 2026', 739676),
 ('baly-byni', 'Май 2026', 739737),
 ('tyzi-lyra', 'январь 2026', 739617),
 ('wydo-qunu', 'Апрель 2026', 739707),
 ('jeme-kosi', 'Май 2026', 739737),
 ('busi-tica', 'июнь 2026', 739768),
 ('xaco-goca', 'Апрель 2026', 739707),
 ('rako-saze', 'март 2026', 739676),
 ('bysu-bato', 'Февраль_2026', 739648),
 ('pife-byvu', 'Февраль 2026', 739648),
 ('gejo-lela', '2026', 739617),
 ('viqu-cajy', 'Январь 2026', 739617)]

## Stage 3 — replace tags with ids

In [7]:
replaced = p.replace()
h8 = next(iter(replaced))
print(replaced[h8][:1200])

## Лист: Расчет взносов

| РАСЧЕТ СТРАХОВЫХ ВЗНОСОВ |  |  |  |
| --- | --- | --- | --- |
| Документ: <e id="dere-gidy">ВЗН-2026-02</e> |  |  |  |
| Проект: <e id="cusa-zypu">24/03/15-01</e> \| Период: <e id="vybi-jiku">Февраль 2026</e> |  |  |  |
|  |  |  |  |
| База начислений, руб. | Начислено взносов, руб. | Код статьи | Комментарий |
| <e id="nyce-jamo">1,117,000.00</e> | <e id="hela-gaqu">335,100.00</e> | <e id="vexa-refo">0300</e> | Сводное начисление за месяц с учетом состава выплат и применимых тарифов |




## Stage 4 — scope checks (contract / period)

In [8]:
scope = p.scope()
print("target label: ", scope.get("target_label"))
print("target row entities:", len(scope.get("target_row") or []),
      "(rest of the card becomes trusted constants)")
print("oos:", {k: v["reasons"] for k, v in scope["oos"].items()})
scope.get("target_row")

target label:  3 этап, 2026 г.
target row entities: 17 (rest of the card becomes trusted constants)
oos: {'2700be8d': ['contract'], '822de338': ['period']}


['bagu-tudi',
 'ceme-weka',
 'diha-zaci',
 'dyxi-zalo',
 'foge-kihi',
 'gejo-toto',
 'lobo-geja',
 'mani-qyxo',
 'popo-nymi',
 'qyqu-niki',
 'roni-myve',
 'sany-xewy',
 'saqu-mofi',
 'tejy-fitu',
 'vujy-daca',
 'xeki-hadi',
 'xepa-gixu']

## Stage 5 — relation extraction (LLM, whole corpus)

In [9]:
# rel_model is already set to the tag model in the config cell above
# (the old "~deepseek/deepseek-pro-latest" id 404s on this account).
# Switch here if you want to A/B another model for relation extraction:
# p.cfg.rel_model = "deepseek/deepseek-v4.1-flash-20261017"

In [10]:
relations_raw = p.relations()
len(relations_raw), relations_raw[:5]

(112,
 [{'equation': 'cusa-zypu == "24/03/15-01"',
   'tag': 'contract',
   'source': 'scope'},
  {'equation': '01.01.2026 <= vybi-jiku && vybi-jiku <= 31.05.2026',
   'tag': 'period',
   'source': 'scope'},
  {'equation': 'hezy-cefe == "24/03/15-01"',
   'tag': 'contract',
   'source': 'scope'},
  {'equation': '01.01.2026 <= xany-mebu && xany-mebu <= 31.05.2026',
   'tag': 'period',
   'source': 'scope'},
  {'equation': 'zuro-novo == "24/03/15-01"',
   'tag': 'contract',
   'source': 'scope'}])

## Stage 6 — solve the system

In [11]:
rels, fishy, verdicts, dangling = p.solve()   # scope -> only target row stays variable
import pandas as pd
rows = [{"i": r.index, "equation": r.equation, "tag": r.tag,
         "status": r.status, "touches_card": r.touches_card,
         "error": r.error} for r in rels]
df = pd.DataFrame(rows)
print(df["status"].value_counts().to_dict(), "| card-touching:",
      df[df.touches_card]["status"].value_counts().to_dict())
df[df.status != "true"]

{'true': 106, 'false': 5} | card-touching: {'true': 59, 'false': 2}


,i,equation,tag,status,touches_card,error
4,4,"zuro-novo == ""24/03/15-01""",contract,false,False,string identity comparison is false
21,21,01.01.2026 <= cavu-fyke && cavu-fyke <= 31.05....,period,false,False,contradicts documents or accepted relations
22,22,01.01.2026 <= busi-tica && busi-tica <= 31.05....,period,false,False,contradicts documents or accepted relations
48,48,rasa-vyto == zuse-nuhy + nune-kose * 0,agg,false,True,contradicts documents or accepted relations
72,72,gapa-hiwa == tejy-fitu + nune-kose * 0,agg,false,True,contradicts documents or accepted relations


## Stage 7 — report

In [12]:
report = p.report()
from checkier.report import format_report_text
print(format_report_text(report, p._mapping))

=== SUMMARY ===
card cells: 17 total | 16 ok | 0 mismatch | 1 unverified
relations: 111 total | 106 true | 5 false | 0 unverifiable
files: 1 duplicates | 2 out-of-scope | 2 dangling | 1 fishy

=== CARD CELL MISMATCHES (0) ===

=== CARD CELLS UNVERIFIED — no equation pinned them down (1) ===
  [number] '30' (id=qyqu-niki, file=card/01_Карточка_учета_фактических_затрат_этап_3.pdf.md)  stated=30

=== RELATIONS NOT PROVEN TRUE — touching card (2 of 111) ===
  #48 FALSE [agg] rasa-vyto == zuse-nuhy + nune-kose * 0
      where: rasa-vyto = '37 272,94' [amount = 37272.94, card/01_Карточка_учета_фактических_затрат_этап_3.pdf.md]; zuse-nuhy = '37 269,26' [amount = 37269.26, card/01_Карточка_учета_фактических_затрат_этап_3.pdf.md]; nune-kose = '3,68' [number = 3.68, card/01_Карточка_учета_фактических_затрат_этап_3.pdf.md]
      error: contradicts documents or accepted relations
      files: card/01_Карточка_учета_фактических_затрат_этап_3.pdf.md
  #72 FALSE [agg] gapa-hiwa == tejy-fitu + nune-ko

In [13]:
import pandas as pd
pd.DataFrame(report["card"]["cells"])

,id,num,mnemonic,raw,type,status,stated,expected,diff,file
0,qyqu-niki,11,qyqu-niki,30,number,unverified,30,NaN,None,0b8a21fe
1,dyxi-zalo,12,dyxi-zalo,мая 2026,date,ok,739737,739737,None,0b8a21fe
2,sany-xewy,66,sany-xewy,"34,14",number,ok,34.14,34.14,None,0b8a21fe
3,vujy-daca,67,vujy-daca,"0,00",number,ok,0.00,0,None,0b8a21fe
4,roni-myve,68,roni-myve,"5 558 474,83",amount,ok,5558474.83,5558474.83,None,0b8a21fe
5,mani-qyxo,69,mani-qyxo,"5 248 019,04",amount,ok,5248019.04,5248019.04,None,0b8a21fe
6,lobo-geja,70,lobo-geja,"5 054 470,26",amount,ok,5054470.26,5054470.26,None,0b8a21fe
7,diha-zaci,71,diha-zaci,"193 548,78",amount,ok,193548.78,193548.78,None,0b8a21fe
8,gejo-toto,72,gejo-toto,"310 455,79",amount,ok,310455.79,310455.79,None,0b8a21fe
9,ceme-weka,73,ceme-weka,"1 671 359,96",amount,ok,1671359.96,1671359.96,None,0b8a21fe


In [14]:
# raw JSON summaries if you need them programmatically
import json
print(json.dumps(report["summary"], indent=2, ensure_ascii=False))

{
  "card_cells_total": 17,
  "ok": 16,
  "mismatch": 0,
  "unverified": 1,
  "duplicates": 1,
  "out_of_scope": 2,
  "dangling": 2,
  "fishy": 1,
  "relations_total": 111,
  "relations_true": 106,
  "relations_false": 5,
  "relations_false_on_card": 2,
  "relations_unverifiable": 0
}
